In [3]:
# Step 5, cell 1: sign in and connect to the new BigQuery project
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
PROJECT_ID = "complaints-ai-analytics"
client = bigquery.Client(project=PROJECT_ID)

ds = bigquery.Dataset(f"{PROJECT_ID}.raw")
ds.location = "US"
client.create_dataset(ds, exists_ok=True)
print("Connected. Datasets:", [d.dataset_id for d in client.list_datasets()])

Connected. Datasets: ['raw']


In [4]:
# Step 5, cell 2: download each archive file, keep every product except credit reporting, load to BigQuery.
# Also builds a small monthly summary of ALL products for context.
import os, zipfile, urllib.request
import pandas as pd

base = "https://files.consumerfinance.gov/f/documents/"
files = [
    "CCDB_Export_5_September_2023_through_March_2024.zip",
    "CCDB_Export_6_April_2024_through_July_2024.zip",
    "CCDB_Export_7_August_2024_through_October_2024.zip",
    "CCDB_Export_8_November_2024_through_December_2024.zip",
    "CCDB_Export_9_January_2025_through_February_2025.zip",
    "CCDB_Export_10_March_2025_through_April_2025.zip",
    "CCDB_Export_11_May_2025_through_June_2025.zip",
    "CCDB_Export_12_July_2025_through_August_2025.zip",
    "CCDB_Export_13_September_2025_through_October_2025.zip",
    "CCDB_Export_14_November_2025_through_December_2025.zip",
    "CCDB_Export_15_January_2026_through_February_2026.zip",
    "CCDB_Export_16_March_2026.zip",
    "CCDB_Export_17_April_2026.zip",
    "CCDB_Export_18_May_2026.zip",
    "CCDB_Export_19_June_2026.zip",
    "CCDB_Export_20_July_2026.zip",
    "CCDB_Export_21_August_2026.zip",
]

# BigQuery column names can't contain spaces or "?", so rename them (same meaning, same order)
rename = {
    "Date received": "date_received",
    "Product": "product",
    "Sub-product": "sub_product",
    "Issue": "issue",
    "Sub-issue": "sub_issue",
    "Consumer complaint narrative": "consumer_complaint_narrative",
    "Company public response": "company_public_response",
    "Company": "company",
    "State": "state",
    "ZIP code": "zip_code",
    "Tags": "tags",
    "Submitted via": "submitted_via",
    "Date sent to company": "date_sent_to_company",
    "Company response to consumer": "company_response_to_consumer",
    "Timely response?": "timely_response",
    "Complaint ID": "complaint_id",
}
cols = list(rename.values())
schema = [bigquery.SchemaField(c, "STRING") for c in cols + ["source_file"]]
table_id = f"{PROJECT_ID}.raw.complaints"

first = True
file_log, summary_parts = [], []

for f in files:
    urllib.request.urlretrieve(base + f, "tmp.zip")
    z = zipfile.ZipFile("tmp.zip")
    csv = [n for n in z.namelist() if n.lower().endswith(".csv")][0]
    total = loaded = 0
    extra_cols = set()

    # keep_default_na=False keeps text like "NA" as text instead of turning it into a blank
    for chunk in pd.read_csv(z.open(csv), dtype=str, chunksize=300_000, keep_default_na=False):
        total += len(chunk)
        extra_cols |= set(chunk.columns) - set(rename)
        chunk = chunk.rename(columns=rename)
        missing = set(cols) - set(chunk.columns)
        if missing:
            raise ValueError(f"{f} is missing columns: {missing}")
        chunk = chunk[cols].replace("", None)
        chunk["source_file"] = f.replace(".zip", "")

        # monthly summary of ALL products (for context)
        month = pd.to_datetime(chunk["date_received"], errors="coerce").dt.strftime("%Y-%m")
        story = chunk["consumer_complaint_narrative"].notna()
        summary_parts.append(
            pd.DataFrame({"month": month, "product": chunk["product"], "story": story})
            .groupby(["month", "product"], dropna=False)
            .agg(complaints=("story", "size"), with_story=("story", "sum"))
            .reset_index()
        )

        # keep everything except credit reporting
        keep = chunk[~chunk["product"].fillna("").str.startswith("Credit reporting")]
        loaded += len(keep)
        if len(keep):
            job_config = bigquery.LoadJobConfig(
                schema=schema,
                write_disposition="WRITE_TRUNCATE" if first else "WRITE_APPEND",
            )
            client.load_table_from_dataframe(keep, table_id, job_config=job_config).result()
            first = False

    file_log.append({"file": f.replace(".zip", ""), "rows_in_file": total,
                     "credit_reporting_excluded": total - loaded, "rows_loaded": loaded,
                     "extra_columns_dropped": ", ".join(sorted(extra_cols)) or "none"})
    print(f"Done: {f}  rows {total:,}  loaded {loaded:,}")
    os.remove("tmp.zip")

# load the monthly summary of all products
summary = (pd.concat(summary_parts)
           .groupby(["month", "product"], dropna=False, as_index=False)[["complaints", "with_story"]].sum())
summary["with_story"] = summary["with_story"].astype(int)
client.load_table_from_dataframe(
    summary, f"{PROJECT_ID}.raw.complaints_monthly_all_products",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
).result()

file_log = pd.DataFrame(file_log)
print()
print(file_log.to_string(index=False))
print()
print(f"TOTAL rows in files: {file_log.rows_in_file.sum():,}   loaded: {file_log.rows_loaded.sum():,}   "
      f"credit reporting excluded: {file_log.credit_reporting_excluded.sum():,}")

Done: CCDB_Export_5_September_2023_through_March_2024.zip  rows 945,532  loaded 164,073
Done: CCDB_Export_6_April_2024_through_July_2024.zip  rows 839,903  loaded 116,883
Done: CCDB_Export_7_August_2024_through_October_2024.zip  rows 824,514  loaded 103,761
Done: CCDB_Export_8_November_2024_through_December_2024.zip  rows 591,058  loaded 69,386
Done: CCDB_Export_9_January_2025_through_February_2025.zip  rows 811,019  loaded 151,319
Done: CCDB_Export_10_March_2025_through_April_2025.zip  rows 803,073  loaded 85,021
Done: CCDB_Export_11_May_2025_through_June_2025.zip  rows 827,343  loaded 88,874
Done: CCDB_Export_12_July_2025_through_August_2025.zip  rows 968,761  loaded 100,030
Done: CCDB_Export_13_September_2025_through_October_2025.zip  rows 1,023,636  loaded 101,794
Done: CCDB_Export_14_November_2025_through_December_2025.zip  rows 1,009,132  loaded 105,624
Done: CCDB_Export_15_January_2026_through_February_2026.zip  rows 1,038,383  loaded 114,613
Done: CCDB_Export_16_March_2026.zip 

In [5]:
# Step 5, cell 3: check the loaded table matches what Python loaded
q = f"""
SELECT
  COUNT(*) AS rows_loaded,
  COUNT(DISTINCT complaint_id) AS unique_ids,
  MIN(date_received) AS first_date,
  MAX(date_received) AS last_date,
  COUNTIF(consumer_complaint_narrative IS NOT NULL) AS with_story,
  COUNT(DISTINCT product) AS products
FROM `{table_id}`
"""
print(client.query(q).to_dataframe().T)
print()
print("Rows loaded by Python:", f"{file_log.rows_loaded.sum():,}")
t = client.get_table(table_id)
print("Table size in BigQuery (GB):", round(t.num_bytes / 1e9, 2))

                      0
rows_loaded     1532691
unique_ids      1532691
first_date   2023-09-01
last_date    2026-08-31
with_story       658136
products             10

Rows loaded by Python: 1,532,691
Table size in BigQuery (GB): 1.2
